<a href="https://colab.research.google.com/github/AngelD222/TFM_2025-2026_algoritmos_recomendacion/blob/main/archivos/Algoritmos_1_2_3_unificado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Algoritmos de recomendación sobre una red social sintética**

Notebook con los Algoritmos 1, 2 y 3 del TFM: topológico (PPR), semántico
(NLP) e híbrido. El modelo híbrido se presenta primero en su versión de
**tres dimensiones** (topología + semántica + personalidad) y después se
**amplía a una cuarta dimensión** de intereses declarados. Ambos grids
comparten exactamente el mismo grafo, los mismos embeddings y el mismo
split, de modo que sus resultados son directamente comparables.
Se cierra con la evaluación offline y la comparativa entre algoritmos.
El Algoritmo 4 (red neuronal) se desarrolla en su propio notebook.

## **1. Setup, carga de datos y limpieza**

Cargamos los cuatro CSV exportados por el simulador y fijamos la semilla.
Renombramos `message_text;` a `message_text`.

In [ ]:
!pip install -q sentence-transformers

In [ ]:
import math
import random
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

# Reproducibilidad
SEED = 2908
random.seed(SEED)
np.random.seed(SEED)

# messages e interactions son los archivos extendidos

ids = {
    'users':        '1Q00bSqmuaYMaPCC9T3ZWPSssOj8ULQAN',
    'friendships':  '1cZY1aGP-ydnmy14IzU1FVsnx9AhocmN6',
    'interactions': '1ZP2Q1n53lmpQ578nchEZbJn-_fbLhvVS',
    'messages':     '1SYLiH-ySCqRDLTxpuzXgrrxKFoDXupe9',
}

def url(file_id):
    return f'https://drive.google.com/uc?export=download&id={file_id}'

import os
def _load_csv(nombre_local, file_id):
    # Carga local si el CSV está presente (ejecución offline); si no, desde Drive.
    src = nombre_local if os.path.exists(nombre_local) else url(file_id)
    return pd.read_csv(src, sep=',', quotechar='"', on_bad_lines='warn')

df_users        = _load_csv('users.csv',        ids['users'])
df_friendships  = _load_csv('friendships.csv',  ids['friendships'])
df_interactions = _load_csv('interactions.csv', ids['interactions'])
df_messages     = _load_csv('messages.csv',     ids['messages'])

# Renombrado de columna mal parseada
if 'message_text;' in df_interactions.columns:
    df_interactions = df_interactions.rename(columns={'message_text;': 'message_text'})

# Normalización de textos (minúsculas, espacios, puntuación residual)
df_messages['content']          = df_messages['content'].astype(str).str.strip().str.lower().str.rstrip(';')
df_interactions['message_text'] = df_interactions['message_text'].astype(str).str.strip().str.lower().str.rstrip(';')

print(f"Usuarios: {len(df_users)}")
print(f"Amistades: {len(df_friendships)}")
print(f"Interacciones: {len(df_interactions)}")
print(f"Mensajes: {len(df_messages)}")

## **2. Infraestructura común entre los algoritmos**

Definimos tres construcciones que reutilizan los tres algoritmos: el grafo ponderado de relaciones,
el diccionario de personalidad por usuario y el mapa de embeddings semánticos.

### ***2.1. Grafo ponderado de relaciones***

Cada usuario es un nodo y el peso base de una amistad es 1.0. Sobre esa capa estática, sumamos una capa de interacciones con pesos crecientes según el coste cognitivo de la acción (`read=0.1`, `like=0.2`, `reply=0.4`, `share=0.6`). Si A interactúa con B pero no le sigue, creamos un enlace implícito con la mitad del peso.

La función `build_weighted_graph` acepta cualquier subconjunto de interacciones, así nos sirve tanto para construir el grafo completo `G` (con todas las interacciones) como, más tarde, para construir el `G_train` (sólo con las interacciones del 80% de entrenamiento).

In [ ]:
# Pesos por tipo de interacción según el coste cognitivo de la acción.
# Las magnitudes concretas se validan en el análisis de sensibilidad (sección 4).
interaction_weights = {'read': 0.1, 'like': 0.2, 'reply': 0.4, 'share': 0.6}

def build_weighted_graph(df_users, df_friendships, df_interactions_subset, weights):
    G = nx.DiGraph()
    G.add_nodes_from(df_users['username'].dropna().unique())

    # Capa estática de amistades
    edges = [
        (row['follower'], row['followed'], 1.0)
        for _, row in df_friendships.iterrows()
        if pd.notna(row['follower']) and pd.notna(row['followed'])
    ]
    G.add_weighted_edges_from(edges)

    # Capa de interacciones (pesada y posiblemente implícita)
    df = df_interactions_subset.copy()
    df['w'] = df['interaction_type'].map(weights).fillna(0)
    grouped = df.groupby(['username', 'message_author'])['w'].sum().reset_index()

    for _, row in grouped.iterrows():
        u, v, w = row['username'], row['message_author'], row['w']
        if pd.isna(u) or pd.isna(v):
            continue
        if G.has_edge(u, v):
            G[u][v]['weight'] += w
        else:
            G.add_edge(u, v, weight=w * 0.5)
    return G

# Grafo completo (sólo para los tests iniciales de los algoritmos)
G = build_weighted_graph(df_users, df_friendships, df_interactions, interaction_weights)
print(f"G: {G.number_of_nodes()} nodos, {G.number_of_edges()} aristas")

### ***2.2. Vectorización de personalidad***

Cada usuario tiene un vector en $\{-1, 0, +1\}^5$ con los cinco rasgos del modelo Big Five
adaptado del simulador: Intellectual, Scrupulousness, Sociability, Friendliness, Neuroticism.

In [ ]:
RASGOS = ['Intellectual', 'Scrupulousness', 'Sociability', 'Friendliness', 'Neuroticism']

def parse_traits_to_vector(trait_string):
    if pd.isna(trait_string):
        return np.zeros(5)
    vector = np.zeros(5)
    ts = str(trait_string)
    for idx, nombre in enumerate(RASGOS):
        if f'{nombre}+' in ts or f'{nombre} +' in ts:
            vector[idx] = 1
        elif f'{nombre}-' in ts or f'{nombre} -' in ts:
            vector[idx] = -1
    return vector

df_users['traits_vector'] = df_users['traits'].apply(parse_traits_to_vector)
personality_dict = dict(zip(df_users['username'], df_users['traits_vector']))

# Diagnóstico rápido del sesgo de la red por rasgo
import collections
for i, name in enumerate(RASGOS):
    valores = [v[i] for v in personality_dict.values()]
    counts = collections.Counter(valores)
    print(f"{name:18s} -> {dict(counts)}")

***Para añadir una dimensión al algoritmo 3:***

In [ ]:
import ast

model = SentenceTransformer('all-MiniLM-L6-v2')

def parse_interests(raw):
    """Lista de intereses declarados. En este corpus cada usuario tiene uno solo,
    pero dejamos parseo de lista por generalidad."""
    if pd.isna(raw):
        return []
    try:
        return ast.literal_eval(raw)
    except (ValueError, SyntaxError):
        return []

df_users['interests_list'] = df_users['interests'].apply(parse_interests)

# Vectorizamos cada etiqueta única UNA SOLA VEZ (hay 35 en el corpus)
all_interests = sorted({tag for lst in df_users['interests_list'] for tag in lst})
interest_emb_cache = dict(zip(all_interests, model.encode(all_interests)))

# El embedding de intereses de un usuario es la media de los embeddings de sus etiquetas.
def build_user_interest_embedding(tags):
    if not tags:
        return None
    return np.mean([interest_emb_cache[t] for t in tags], axis=0)

interests_dict = {
    row['username']: build_user_interest_embedding(row['interests_list'])
    for _, row in df_users.iterrows()
}

n_con_interes = sum(1 for v in interests_dict.values() if v is not None)
print(f"Etiquetas únicas: {len(all_interests)}")
print(f"Usuarios con interés declarado: {n_con_interes} / {len(df_users)}")

### ***2.3. Mapa de embeddings semánticos***

Vectorizamos una sola vez todos los textos únicos del histórico y del catálogo de candidatos
con `all-MiniLM-L6-v2` (el mismo modelo que usa el simulador). El mapa indexado por texto
permite acceder a cualquier embedding en $O(1)$ durante la evaluación.

In [ ]:


candidate_texts_unique = df_messages['content'].dropna().drop_duplicates().tolist()
history_texts_unique   = df_interactions['message_text'].dropna().drop_duplicates().tolist()

print(f"Calculando embeddings ({len(candidate_texts_unique)} candidatos + {len(history_texts_unique)} históricos)...")
candidate_embs = model.encode(candidate_texts_unique, show_progress_bar=True)
history_embs   = model.encode(history_texts_unique,   show_progress_bar=True)

embedding_map = {
    **dict(zip(history_texts_unique, history_embs)),
    **dict(zip(candidate_texts_unique, candidate_embs)),
}
print(f"Mapa de embeddings listo: {len(embedding_map)} textos únicos.")

def get_embeddings(texts, embedding_map, model):
    """Devuelve embeddings desde el mapa, calculando sobre la marcha si falta alguno."""
    result = [None] * len(texts)
    missing, missing_idx = [], []
    for i, t in enumerate(texts):
        if t in embedding_map:
            result[i] = embedding_map[t]
        else:
            missing.append(t); missing_idx.append(i)
    if missing:
        fresh = model.encode(missing)
        for idx, emb in zip(missing_idx, fresh):
            result[idx] = emb
    return np.array(result)

## **3. Definición de los tres algoritmos**

Cada algoritmo se define como una función reutilizable, parametrizada por el dataframe
de histórico (`df_history`). Esto permite usar las mismas funciones tanto para los tests
iniciales (sobre toda la red) como para la evaluación offline (sobre `df_train`).

### ***3.1. Algoritmo 1 — recomendación de usuarios basado en grafos ponderados por PPR***

Personalized PageRank sobre el grafo ponderado. La función acepta opcionalmente un `ppr_cache`
con los scores precomputados, lo cual evita recalcular PPR en cada llamada durante la evaluación
masiva.

In [ ]:
def compute_ppr(G, target_user, alpha=0.85):
    """Devuelve el dict de scores PPR centrado en target_user."""
    personalization = {n: 0.0 for n in G.nodes()}
    if target_user in G:
        personalization[target_user] = 1.0
    return nx.pagerank(G, alpha=alpha, personalization=personalization, weight='weight')


def get_ppr_recommendations(G, target_user, top_k=10, ppr_cache=None):
    if target_user not in G:
        return []
    if ppr_cache is not None and target_user in ppr_cache:
        ppr_scores = ppr_cache[target_user]
    else:
        ppr_scores = compute_ppr(G, target_user)
    already_followed = set(G.successors(target_user)) | {target_user}
    candidates = {n: s for n, s in ppr_scores.items() if n not in already_followed}
    return sorted(candidates.items(), key=lambda x: x[1], reverse=True)[:top_k]


# Test rápido con un usuario aleatorio
test_user = random.choice(list(G.nodes()))
recs = get_ppr_recommendations(G, test_user, top_k=5)
print(f"PPR para {test_user}:")
for rank, (user, score) in enumerate(recs, 1):
    print(f"  {rank}. {user}  ({score:.6f})")

En el algoritmo PageRank la suma de las puntuaciones de todos los nodos de la red es exactamente 1.0. Con una red de 220 usuarios, si todos tuvieran exactamente la misma importancia, la puntuación base de cada uno sería $1 / 220 = 0.0045$.

### ***3.2. Algoritmo 2 — filtrado por contenido mediante NLP***

Construye un perfil semántico del usuario como media de los embeddings de su histórico y recomienda los mensajes más similares en el espacio vectorial. Lo que vamos a hacer es:

1. Extraeremos los textos de los mensajes con los que el usuario ha interactuado en el pasado (interactions.csv)

2. Convertiremos esos textos en embeddings de 384 dimensiones usando el modelo all-MiniLM-L6-v2 (el mismo que se usa en el simulador)

3. Calcularemos el centroide (la media matemática) de esos vectores. Este será el perfil de intereses del usuario

4. Calcularemos la similitud del coseno entre ese perfil y todos los posts disponibles en messages.csv que el usuario aún no haya leído.

In [ ]:
def get_semantic_recommendations(target_user, df_history, df_candidates, embedding_map, model, top_k=10):
    history = df_history[df_history['username'] == target_user]['message_text'].dropna().tolist()
    if not history:
        return []

    history_set = set(history)

    candidates_df = df_candidates[
        ~df_candidates['content'].isin(history_set) &
        (df_candidates['sender_username'] != target_user)
    ].drop_duplicates('content').copy()

    if candidates_df.empty:
        return []

    history_embs   = get_embeddings(history, embedding_map, model)
    candidate_embs = get_embeddings(candidates_df['content'].tolist(), embedding_map, model)

    user_profile = np.mean(history_embs, axis=0).reshape(1, -1)
    similarities = cosine_similarity(user_profile, candidate_embs)[0]

    candidates_df['score'] = similarities
    top = candidates_df.nlargest(top_k, 'score')
    return [
        {'author': row['sender_username'], 'content': row['content'], 'score': float(row['score'])}
        for _, row in top.iterrows()
    ]


# Test rápido
test_user = df_interactions['username'].dropna().iloc[13]
recs = get_semantic_recommendations(test_user, df_interactions, df_messages, embedding_map, model, top_k=5)
print(f"Recomendaciones semánticas para {test_user}:")
for rank, rec in enumerate(recs, 1):
    preview = rec['content'][:80] + ('...' if len(rec['content']) > 80 else '')
    print(f"  {rank}. [{rec['score']:.3f}] {rec['author']}: {preview}")

Al variar el `target_user` se observa que prácticamente siempre las recomendaciones giran en
torno al conflicto de Ucrania. Verificamos cuánto pesa esta temática en el catálogo para confirmar que el sesgo no es ruido del algoritmo sino del corpus.

In [ ]:

# Aquí ponemos palabras relacionadas con el tema que se repite.
palabras_clave = ['ukraine', 'war', 'peace', 'russia', 'conflict']
# Unimos las palabras con el símbolo '|' que significa "O" (ej: ukraine O war O peace)
patron = '|'.join(palabras_clave)

# Creamos el filtro buscando en la columna de contenido
# case=False hace que no importe si está en mayúsculas o minúsculas
# na=False evita que el código se rompa si hay mensajes vacíos
filtro = df_messages['content'].str.contains(patron, case=False, na=False)

print(f"Mensajes totales: {len(df_messages)}")
print(f"Mensajes sobre la temática Ucrania/Rusia/guerra: {filtro.sum()}")
print(f"Porcentaje de dominancia: {filtro.mean()*100:.2f}%")

### **3.3. Algoritmo 3 — modelo híbrido con topología + semántica + personalidad + intereses**

Combinación lineal ponderada de tres dimensiones: topológica (PPR), semántica (similitud del coseno entre perfil y candidato) y psicológica (similitud del coseno entre vectores Big Five). Acepta `ppr_cache` igual que el Algoritmo 1.

Para que la ecuación matemática que suma el grafo, la semántica y la personalidad funcione correctamente tenemos que ajustar primero dos cosas:

- El Algoritmo 1 devuelve valores muy pequeños ($0.02$), mientras que la similitud del coseno del Algoritmo 2 devuelve valores entre $[-1, 1]$. Si los sumamos sin nromalizar, la semántica ganará siempre al grafo. Debemos normalizar todo al rango $[0, 1]$.

- Datos no numéricos: la personalidad y los rasgos de los usuarios vienen en cadenas de texto en la columna traits ("Con+, Neu+"). Debemos extraer esta información basada en el modelo de los Big Five y convertirla en vectores para poder aplicar operaciones matemáticas de homofilia.

Empezamos con la versión de **tres dimensiones** (topología, semántica y
personalidad), evaluándola primero con unos pesos por defecto arbitrarios
($\alpha=0.4, \beta=0.3, \gamma=0.3$) y calibrándola
después por grid search. Más adelante activamos la **cuarta dimensión** de
intereses declarados ($\delta>0$) y repetimos el grid en 4D. La misma
función `get_hybrid_recommendations` cubre ambos casos: con `delta=0` y sin
`interests_dict` se comporta exactamente como el modelo de tres dimensiones.

In [ ]:
def get_hybrid_recommendations(target_user, df_history, df_candidates, G,
                               embedding_map, model, personality_dict,
                               interests_dict=None,
                               ppr_cache=None,
                               alpha=0.4, beta=0.3, gamma=0.3, delta=0.0,
                               top_k=10):
    history = df_history[df_history['username'] == target_user]['message_text'].dropna().tolist()
    if not history:
        return []

    history_set = set(history)
    candidates_df = df_candidates[
        ~df_candidates['content'].isin(history_set) &
        (df_candidates['sender_username'] != target_user)
    ].drop_duplicates('content').copy()

    if candidates_df.empty:
        return []

    # --- DIMENSIÓN 1: topología (PPR) ---
    ppr_scores = (ppr_cache[target_user]
                  if ppr_cache is not None and target_user in ppr_cache
                  else compute_ppr(G, target_user))
    candidate_authors = candidates_df['sender_username'].unique()
    ppr_for_candidates = {a: ppr_scores.get(a, 0.0) for a in candidate_authors}

    if ppr_for_candidates:
        vals = list(ppr_for_candidates.values())
        min_p, max_p = min(vals), max(vals)
        rango = max_p - min_p
        norm_ppr = ({a: (v - min_p) / rango for a, v in ppr_for_candidates.items()}
                    if rango > 0 else {a: 0.0 for a in ppr_for_candidates})
    else:
        norm_ppr = {}

    # --- DIMENSIÓN 2: semántica (preferencia revelada por el histórico) ---
    history_embs   = get_embeddings(history, embedding_map, model)
    candidate_embs = get_embeddings(candidates_df['content'].tolist(), embedding_map, model)
    user_profile = np.mean(history_embs, axis=0).reshape(1, -1)
    semantic_sims = (cosine_similarity(user_profile, candidate_embs)[0] + 1) / 2

    # --- DIMENSIÓN 3: personalidad ---
    target_pers = personality_dict.get(target_user, np.zeros(5)).reshape(1, -1)

    # --- DIMENSIÓN 4: intereses declarados (preferencia explícita) ---
    user_int_emb = (interests_dict.get(target_user)
                    if interests_dict is not None else None)
    if user_int_emb is not None:
        interest_sims = (cosine_similarity(
            user_int_emb.reshape(1, -1), candidate_embs)[0] + 1) / 2
    else:
        # Si el usuario no declara intereses, neutralizamos esta dimensión.
        interest_sims = np.full(len(candidates_df), 0.5)

    results = []
    for idx, row in enumerate(candidates_df.itertuples()):
        author = row.sender_username
        s_g = norm_ppr.get(author, 0.0)
        s_s = semantic_sims[idx]
        author_pers = personality_dict.get(author, np.zeros(5)).reshape(1, -1)
        s_p = (cosine_similarity(target_pers, author_pers)[0][0] + 1) / 2
        s_i = interest_sims[idx]

        results.append({
            'author':      author,
            'content':     row.content,
            'scores':      (s_g, s_s, s_p, s_i),
            'final_score': alpha * s_g + beta * s_s + gamma * s_p + delta * s_i,
        })

    return sorted(results, key=lambda x: x['final_score'], reverse=True)[:top_k]

## **4. Evaluación offline**



Definimos el protocolo común a todos los algoritmos: split temporal por usuario, grafo de entrenamiento `G_train` que ignora las interacciones del test, métricas Recall@10, NDCG@10 y diversidad temática.

### ***4.1. Split temporal y G_train***

Para que la evaluación sea robusta no podemos evaluar a usuarios que solo tienen 1 o 2 interacciones (porque si el 80% va a entrenamiento, nos quedaríamos con 0.8 interacciones, lo cual es inmanejable). Por esto, filtramos usuarios con al menos cuatro interacciones (umbral mínimo para que el split sea válido) y asignamos el 80% más antiguo a entrenamiento, el 20% más reciente a prueba. Reconstruimos el grafo usando solo las interacciones de entrenamiento para evitar fuga de información temporal.

In [ ]:
df_interactions['created_at'] = pd.to_datetime(df_interactions['created_at'])
df_interactions = df_interactions.sort_values(['username', 'created_at'])

min_interactions = 4
user_counts = df_interactions['username'].value_counts()
valid_users = user_counts[user_counts >= min_interactions].index
df_eval = df_interactions[df_interactions['username'].isin(valid_users)].copy()

df_eval['rank']  = df_eval.groupby('username')['created_at'].rank(method='first')
df_eval['total'] = df_eval.groupby('username')['username'].transform('count')
df_eval['is_train'] = df_eval['rank'] <= (df_eval['total'] * 0.8)

df_train = df_eval[ df_eval['is_train']].drop(columns=['rank', 'total', 'is_train']).copy()
df_test  = df_eval[~df_eval['is_train']].drop(columns=['rank', 'total', 'is_train']).copy()

print(f"Usuarios evaluables (>= {min_interactions} interacciones): {len(valid_users)}")
print(f"Train: {len(df_train)} interacciones")
print(f"Test:  {len(df_test)} interacciones")

# G_train: grafo ponderado usando solo interacciones de entrenamiento
G_train = build_weighted_graph(df_users, df_friendships, df_train, interaction_weights)
print(f"G_train: {G_train.number_of_nodes()} nodos, {G_train.number_of_edges()} aristas")

### ***4.2. Pre-cálculo de PPR sobre G_train***

Calculamos una sola vez el PPR centrado en cada usuario evaluable. Esto convierte el grid search de horas en minutos: en lugar de recalcular PageRank en cada combinación de pesos, se consulta el diccionario en $O(1)$.

In [ ]:
print("Pre-calculando PPR sobre G_train para cada usuario evaluable...")
ppr_per_user = {}
usuarios_para_ppr = [u for u in valid_users if u in G_train]

for user in tqdm(usuarios_para_ppr):
    ppr_per_user[user] = compute_ppr(G_train, user)

print(f"PPR pre-calculado para {len(ppr_per_user)} usuarios.")

### ***4.3. Métricas***

`Recall@10` y `NDCG@10` operan sobre conjuntos (sin duplicados en el ground truth, para no penalizar artificialmente). La diversidad se calcula como uno menos la similitud del coseno promedio entre todos los pares de embeddings de la lista recomendada.

In [ ]:
def calculate_recall(recommended, true_items, k=10):
    true_set = set(true_items)
    if not true_set: return 0.0
    hits = len(set(recommended[:k]) & true_set)
    return hits / len(true_set)

def calculate_ndcg(recommended, true_items, k=10):
    true_set = set(true_items)
    if not true_set: return 0.0
    rec_k = recommended[:k]
    dcg  = sum(1.0 / math.log2(i + 2) for i, item in enumerate(rec_k) if item in true_set)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(min(len(true_set), k)))
    return dcg / idcg if idcg > 0 else 0.0

def calculate_diversity(texts, embedding_map, model):
    if len(texts) < 2: return 0.0
    embs = get_embeddings(texts, embedding_map, model)
    sim = cosine_similarity(embs)
    triu = sim[np.triu_indices(len(texts), k=1)]
    return 1.0 - float(np.mean(triu))

Validamos empíricamente la mejor combinación de pesos para el algoritmo 1:

In [ ]:
# === SENSIBILIDAD DEL ALGORITMO 1 A LOS PESOS DE INTERACCIÓN ===
df_train_inner = df_train.sort_values(by=['username', 'created_at']).copy()
df_train_inner['rank']  = df_train_inner.groupby('username')['created_at'].rank(method='first')
df_train_inner['total'] = df_train_inner.groupby('username')['username'].transform('count')
df_train_inner['is_inner_train'] = df_train_inner['rank'] <= (df_train_inner['total'] * 0.8)

df_inner_train = df_train_inner[ df_train_inner['is_inner_train']].drop(columns=['rank','total','is_inner_train'])
df_inner_val   = df_train_inner[~df_train_inner['is_inner_train']].drop(columns=['rank','total','is_inner_train'])

configuraciones = [
    {'read': 0.05, 'like': 0.1,  'reply': 0.2, 'share': 0.3 },   # base
    {'read': 0.02, 'like': 0.05, 'reply': 0.1, 'share': 0.2 },   # conservador
    {'read': 0.1,  'like': 0.2,  'reply': 0.4, 'share': 0.6 },   # agresivo
    {'read': 0.05, 'like': 0.15, 'reply': 0.3, 'share': 0.5 },   # share más fuerte
    {'read': 0.05, 'like': 0.1,  'reply': 0.3, 'share': 0.2 },   # reply > share (variante)
    {'read': 0.1,  'like': 0.1,  'reply': 0.1, 'share': 0.1 },   # uniforme (control)
]

resultados_sens = []
for cfg in configuraciones:
    G_inner = build_weighted_graph(df_users, df_friendships, df_inner_train, cfg)
    ppr_inner = {u: compute_ppr(G_inner, u)
                 for u in df_inner_val['username'].dropna().unique() if u in G_inner}

    recalls, ndcgs = [], []
    for user in df_inner_val['username'].dropna().unique():
        true_a = df_inner_val[df_inner_val['username'] == user]['message_author'].dropna().unique().tolist()
        already = set(G_inner.successors(user)) if user in G_inner else set()
        true_new = [a for a in true_a if a not in already and a != user]
        if not true_new or user not in ppr_inner:
            continue
        recs = get_ppr_recommendations(G_inner, user, top_k=10, ppr_cache=ppr_inner)
        rec_users = [r[0] for r in recs]
        recalls.append(calculate_recall(rec_users, true_new))
        ndcgs.append(calculate_ndcg(rec_users, true_new))

    resultados_sens.append({
        'cfg': cfg,
        'recall@10': np.mean(recalls) if recalls else 0.0,
        'ndcg@10':   np.mean(ndcgs)   if ndcgs   else 0.0,
        'n': len(recalls),
    })

print("Sensibilidad del Algoritmo 1 a los pesos (sobre inner-val):")
for r in sorted(resultados_sens, key=lambda x: x['ndcg@10'], reverse=True):
    print(f"  {r['cfg']}  →  Recall={r['recall@10']:.4f}, NDCG={r['ndcg@10']:.4f}  (n={r['n']})")

Adoptamos por tanto {read = 0.1, like = 0.2, reply = 0.4, share = 0.6} como configuración definitiva, que es la que se aplica de aquí en adelante en todos los algoritmos que usan G_train

### ***4.4. Bucle de evaluación con pesos por defecto***

Evaluamos los tres algoritmos sobre `df_test` con los pesos iniciales del Algoritmo 3 ($\alpha=0.4, \beta=0.3, \gamma=0.3). Excluimos del ground truth del Algoritmo 1 los autores que el usuario ya seguía en `G_train`, para evaluar la capacidad de descubrir conexiones nuevas y no la de redescubrir las existentes.

In [ ]:
usuarios_test = df_test['username'].unique()

resultados = {
    'Algoritmo 1 (Topológico)': {'recall': [], 'ndcg': []},
    'Algoritmo 2 (Semántico)':  {'recall': [], 'ndcg': [], 'diversity': []},
    'Algoritmo 3 (Híbrido)':    {'recall': [], 'ndcg': [], 'diversity': []},
}

for user in tqdm(usuarios_test, desc='Evaluando'):
    true_texts        = df_test[df_test['username'] == user]['message_text'].dropna().drop_duplicates().tolist()
    true_authors_raw  = df_test[df_test['username'] == user]['message_author'].dropna().tolist()
    already_followed  = set(G_train.successors(user)) if user in G_train else set()
    true_authors_new  = list({a for a in true_authors_raw if a not in already_followed and a != user})

    # Algoritmo 1
    if true_authors_new:
        recs1 = get_ppr_recommendations(G_train, user, top_k=10, ppr_cache=ppr_per_user)
        if recs1:
            rec_users = [r[0] for r in recs1]
            resultados['Algoritmo 1 (Topológico)']['recall'].append(calculate_recall(rec_users, true_authors_new))
            resultados['Algoritmo 1 (Topológico)']['ndcg'].append(  calculate_ndcg(rec_users,   true_authors_new))

    # Algoritmo 2
    if true_texts:
        recs2 = get_semantic_recommendations(user, df_train, df_messages, embedding_map, model, top_k=10)
        if recs2:
            rec_texts = [r['content'] for r in recs2]
            resultados['Algoritmo 2 (Semántico)']['recall'].append(   calculate_recall(rec_texts, true_texts))
            resultados['Algoritmo 2 (Semántico)']['ndcg'].append(     calculate_ndcg(rec_texts,   true_texts))
            resultados['Algoritmo 2 (Semántico)']['diversity'].append(calculate_diversity(rec_texts, embedding_map, model))

    # Algoritmo 3
    if true_texts:
        recs3 = get_hybrid_recommendations(user, df_train, df_messages, G_train,
                                           embedding_map, model, personality_dict,
                                           ppr_cache=ppr_per_user,
                                           alpha=0.4, beta=0.3, gamma=0.3, delta=0.0, top_k=10)
        if recs3:
            rec_texts = [r['content'] for r in recs3]
            resultados['Algoritmo 3 (Híbrido)']['recall'].append(   calculate_recall(rec_texts, true_texts))
            resultados['Algoritmo 3 (Híbrido)']['ndcg'].append(     calculate_ndcg(rec_texts,   true_texts))
            resultados['Algoritmo 3 (Híbrido)']['diversity'].append(calculate_diversity(rec_texts, embedding_map, model))

print("\n" + "="*55)
print(" RESULTADOS CON PESOS POR DEFECTO 3D (α=0.4, β=0.3, γ=0.3, δ=0)")
print("="*55)
for nombre, m in resultados.items():
    print(f"\n{nombre}:")
    print(f"   Recall@10: {np.mean(m['recall']):.4f}")
    print(f"   NDCG@10:   {np.mean(m['ndcg']):.4f}")
    if 'diversity' in m:
        print(f"   Diversidad: {np.mean(m['diversity']):.4f}")

## **5. Calibración del Algoritmo 3 (3D) sin data leakage**

El grid search debe elegir los pesos $(\alpha, \beta, \gamma)$ **sin mirar**
el conjunto de prueba. Si evaluáramos cada combinación directamente sobre
`df_test` y reportásemos la mejor, el conjunto de prueba habría intervenido
en la selección del modelo, inflando las métricas (*data leakage* por
selección de hiperparámetros).

Para evitarlo construimos un sub-split temporal **interno** dentro de
`df_train` (`df_inner_train` / `df_inner_val`), corremos el grid sobre
`df_inner_val`, y solo cuando tenemos la configuración ganadora la
evaluamos **una única vez** sobre `df_test`. Todas las estructuras
derivadas (grafo, PPR, perfiles) se reconstruyen usando exclusivamente
`df_inner_train` durante la búsqueda. Este mismo inner-val se reutiliza
luego para el grid 4D, garantizando que ambas calibraciones son comparables.

In [ ]:
# === GRID SEARCH 3D SOBRE UN INNER-VAL (sin tocar df_test) ===
# 1) Split temporal interno dentro de df_train
df_tr = df_train.sort_values(['username', 'created_at']).copy()
df_tr['rank']  = df_tr.groupby('username')['created_at'].rank(method='first')
df_tr['total'] = df_tr.groupby('username')['username'].transform('count')
df_tr['is_inner_train'] = df_tr['rank'] <= (df_tr['total'] * 0.8)

df_inner_train = df_tr[ df_tr['is_inner_train']].drop(columns=['rank','total','is_inner_train']).copy()
df_inner_val   = df_tr[~df_tr['is_inner_train']].drop(columns=['rank','total','is_inner_train']).copy()
print(f"Inner-train: {len(df_inner_train)} | Inner-val: {len(df_inner_val)}")

# 2) Estructuras derivadas SOLO del inner-train
G_inner = build_weighted_graph(df_users, df_friendships, df_inner_train, interaction_weights)
ppr_inner = {u: compute_ppr(G_inner, u)
             for u in df_inner_val['username'].dropna().unique() if u in G_inner}
print(f"G_inner: {G_inner.number_of_nodes()} nodos, {G_inner.number_of_edges()} aristas")

# 3) Usuarios del inner-val con ground truth textual
usuarios_inner_val = [
    u for u in df_inner_val['username'].dropna().unique()
    if df_inner_val[df_inner_val['username'] == u]['message_text'].dropna().drop_duplicates().tolist()
    and u in ppr_inner
]

# 4) Grid 3D (delta=0, sin interests_dict) evaluado sobre el inner-val
step = 0.1
valores = np.arange(0.0, 1.0 + step/2, step)
combinaciones_3d = [(round(a,1), round(b,1), round(c,1))
                    for a in valores for b in valores for c in valores
                    if np.isclose(a + b + c, 1.0)]
print(f"Se evaluarán {len(combinaciones_3d)} combinaciones 3D sobre {len(usuarios_inner_val)} usuarios.")

resultados_grid_3d = []
for (alpha, beta, gamma) in tqdm(combinaciones_3d, desc='Grid 3D (inner-val)'):
    ndcg_list, recall_list, div_list = [], [], []
    for user in usuarios_inner_val:
        true_texts = df_inner_val[df_inner_val['username'] == user]['message_text'].dropna().drop_duplicates().tolist()
        recs = get_hybrid_recommendations(user, df_inner_train, df_messages, G_inner,
                                          embedding_map, model, personality_dict,
                                          interests_dict=None, ppr_cache=ppr_inner,
                                          alpha=alpha, beta=beta, gamma=gamma, delta=0.0, top_k=10)
        if recs:
            rec_texts = [r['content'] for r in recs]
            ndcg_list.append(  calculate_ndcg(rec_texts,   true_texts))
            recall_list.append(calculate_recall(rec_texts, true_texts))
            div_list.append(   calculate_diversity(rec_texts, embedding_map, model))
    resultados_grid_3d.append({'alpha': alpha, 'beta': beta, 'gamma': gamma,
        'recall': np.mean(recall_list) if recall_list else 0.0,
        'ndcg':   np.mean(ndcg_list)   if ndcg_list   else 0.0,
        'diversity': np.mean(div_list) if div_list    else 0.0})

resultados_grid_3d.sort(key=lambda x: x['ndcg'], reverse=True)
print("\nTOP-5 CONFIGURACIONES 3D SOBRE INNER-VAL (por NDCG@10)")
for i, r in enumerate(resultados_grid_3d[:5], 1):
    print(f"#{i}  α={r['alpha']}, β={r['beta']}, γ={r['gamma']}  "
          f"→ R={r['recall']:.4f}, NDCG={r['ndcg']:.4f}, Div={r['diversity']:.4f}")
best_3d = resultados_grid_3d[0]
print(f"\n>>> Ganadora 3D (sin ver df_test): α={best_3d['alpha']}, β={best_3d['beta']}, γ={best_3d['gamma']}")

### ***5.1. Evaluación final del Algoritmo 3-3D sobre `df_test`***

Con los pesos fijados sobre el inner-val, evaluamos el modelo 3D una sola
vez sobre `df_test`, reconstruyendo las estructuras con **todo** `df_train`
(`G_train`, `ppr_per_user`). Lo único que nunca ha intervenido en la
elección de pesos es `df_test`.

In [ ]:
# === EVALUACIÓN FINAL DEL ALGORITMO 3 3D SOBRE df_test (una sola vez) ===
a3, b3, g3 = best_3d['alpha'], best_3d['beta'], best_3d['gamma']
usuarios_validos = [u for u in usuarios_test
    if df_test[df_test['username'] == u]['message_text'].dropna().drop_duplicates().tolist()]

recall_3d, ndcg_3d, div_3d = [], [], []
for user in usuarios_validos:
    true_texts = df_test[df_test['username'] == user]['message_text'].dropna().drop_duplicates().tolist()
    recs = get_hybrid_recommendations(user, df_train, df_messages, G_train,
                                      embedding_map, model, personality_dict,
                                      interests_dict=None, ppr_cache=ppr_per_user,
                                      alpha=a3, beta=b3, gamma=g3, delta=0.0, top_k=10)
    if recs:
        rec_texts = [r['content'] for r in recs]
        ndcg_3d.append(  calculate_ndcg(rec_texts,   true_texts))
        recall_3d.append(calculate_recall(rec_texts, true_texts))
        div_3d.append(   calculate_diversity(rec_texts, embedding_map, model))

a3_3d_test = {'alpha': a3, 'beta': b3, 'gamma': g3,
    'recall': np.mean(recall_3d) if recall_3d else 0.0,
    'ndcg':   np.mean(ndcg_3d)   if ndcg_3d   else 0.0,
    'diversity': np.mean(div_3d) if div_3d    else 0.0, 'n': len(recall_3d)}

print("="*60)
print(" ALGORITMO 3 3D — MÉTRICAS FINALES SOBRE df_test")
print("="*60)
print(f"  Pesos (inner-val): α={a3}, β={b3}, γ={g3}")
print(f"  Recall@10:           {a3_3d_test['recall']:.4f}")
print(f"  NDCG@10:             {a3_3d_test['ndcg']:.4f}")
print(f"  Diversidad temática: {a3_3d_test['diversity']:.4f}")
print(f"  Usuarios evaluados:  {a3_3d_test['n']}")
print(f"  (Referencia inner-val: NDCG={best_3d['ndcg']:.4f}, R={best_3d['recall']:.4f})")

## **6. Calibración del Algoritmo 3 con 4D, añadiendo intereses declarados**

El grid search debe elegir los pesos $(\alpha, \beta, \gamma, \delta)$ **sin mirar** el conjunto de prueba. Si evaluáramos cada combinación directamente sobre `df_test` y luego reportásemos la mejor, el conjunto de prueba habría intervenido en la selección del modelo, inflando las métricas (una forma de *data leakage* por selección de hiperparámetros).

Reutilizamos exactamente el mismo inner-val, grafo `G_inner` y caché `ppr_inner` construidos en la sección 5 para el grid 3D, de modo que ambas calibraciones (3D y 4D) son directamente comparables. La única diferencia es que ahora activamos la cuarta dimensión pasando `interests_dict` y dejando que $\delta$ tome valores no nulos.

In [ ]:
# === GRID SEARCH 4D SOBRE EL MISMO INNER-VAL (sin tocar df_test) ===
# Reutilizamos df_inner_train, df_inner_val, G_inner, ppr_inner y usuarios_inner_val
# ya construidos en la sección 5 (grid 3D), para que ambas calibraciones sean comparables.

# 4) Grid 4D evaluado sobre el inner-val
step = 0.1
valores = np.arange(0.0, 1.0 + step/2, step)
combinaciones = [
    (round(a,1), round(b,1), round(g,1), round(d,1))
    for a in valores for b in valores for g in valores for d in valores
    if np.isclose(a + b + g + d, 1.0)
]
print(f"Se evaluarán {len(combinaciones)} combinaciones sobre {len(usuarios_inner_val)} usuarios del inner-val.")

resultados_grid = []
for (alpha, beta, gamma, delta) in tqdm(combinaciones, desc='Grid 4D (inner-val)'):
    ndcg_list, recall_list, div_list = [], [], []
    for user in usuarios_inner_val:
        true_texts = df_inner_val[df_inner_val['username'] == user]['message_text'].dropna().drop_duplicates().tolist()
        recs = get_hybrid_recommendations(
            user, df_inner_train, df_messages, G_inner,
            embedding_map, model, personality_dict,
            interests_dict=interests_dict,
            ppr_cache=ppr_inner,
            alpha=alpha, beta=beta, gamma=gamma, delta=delta, top_k=10
        )
        if recs:
            rec_texts = [r['content'] for r in recs]
            ndcg_list.append(  calculate_ndcg(rec_texts,   true_texts))
            recall_list.append(calculate_recall(rec_texts, true_texts))
            div_list.append(   calculate_diversity(rec_texts, embedding_map, model))

    resultados_grid.append({
        'alpha': alpha, 'beta': beta, 'gamma': gamma, 'delta': delta,
        'recall':    np.mean(recall_list) if recall_list else 0.0,
        'ndcg':      np.mean(ndcg_list)   if ndcg_list   else 0.0,
        'diversity': np.mean(div_list)    if div_list    else 0.0,
    })

resultados_grid.sort(key=lambda x: x['ndcg'], reverse=True)
print("\n" + "="*70)
print(" TOP-10 CONFIGURACIONES SOBRE INNER-VAL (ordenadas por NDCG@10)")
print("="*70)
for i, r in enumerate(resultados_grid[:10], 1):
    print(f"#{i:2d}  α={r['alpha']}, β={r['beta']}, γ={r['gamma']}, δ={r['delta']}  "
          f"→ R={r['recall']:.4f}, NDCG={r['ndcg']:.4f}, Div={r['diversity']:.4f}")

best = resultados_grid[0]
print(f"\n>>> Configuración ganadora (elegida SIN ver df_test): "
      f"α={best['alpha']}, β={best['beta']}, γ={best['gamma']}, δ={best['delta']}")


### ***6.1. Evaluación final de la configuración ganadora sobre `df_test`***

Con los pesos ya fijados sobre el inner-val, evaluamos el Algoritmo 3 una sola vez sobre `df_test`. Aquí sí reconstruimos las estructuras con **todo** `df_train` (vía `G_train` y `ppr_per_user`), porque el conjunto de entrenamiento completo es conocimiento legítimo en el momento de la inferencia final; lo único que nunca ha intervenido en la elección de los pesos es `df_test`.

In [ ]:
# === EVALUACIÓN FINAL DEL ALGORITMO 3 4D SOBRE df_test (una sola vez) ===
a_opt, b_opt, g_opt, d_opt = best['alpha'], best['beta'], best['gamma'], best['delta']

usuarios_validos = [
    u for u in usuarios_test
    if df_test[df_test['username'] == u]['message_text'].dropna().drop_duplicates().tolist()
]

recall_4d, ndcg_4d, div_4d = [], [], []
for user in usuarios_validos:
    true_texts = df_test[df_test['username'] == user]['message_text'].dropna().drop_duplicates().tolist()
    recs = get_hybrid_recommendations(
        user, df_train, df_messages, G_train,
        embedding_map, model, personality_dict,
        interests_dict=interests_dict,
        ppr_cache=ppr_per_user,
        alpha=a_opt, beta=b_opt, gamma=g_opt, delta=d_opt, top_k=10
    )
    if recs:
        rec_texts = [r['content'] for r in recs]
        ndcg_4d.append(  calculate_ndcg(rec_texts,   true_texts))
        recall_4d.append(calculate_recall(rec_texts, true_texts))
        div_4d.append(   calculate_diversity(rec_texts, embedding_map, model))

# Guardamos las métricas finales del modelo ganador para el resumen comparativo
a3_4d_test = {
    'alpha': a_opt, 'beta': b_opt, 'gamma': g_opt, 'delta': d_opt,
    'recall':    np.mean(recall_4d) if recall_4d else 0.0,
    'ndcg':      np.mean(ndcg_4d)   if ndcg_4d   else 0.0,
    'diversity': np.mean(div_4d)    if div_4d    else 0.0,
    'n':         len(recall_4d),
}

print("="*60)
print(" ALGORITMO 3 4D — MÉTRICAS FINALES SOBRE df_test")
print("="*60)
print(f"  Pesos (elegidos en inner-val): "
      f"α={a_opt}, β={b_opt}, γ={g_opt}, δ={d_opt}")
print(f"  Recall@10:           {a3_4d_test['recall']:.4f}")
print(f"  NDCG@10:             {a3_4d_test['ndcg']:.4f}")
print(f"  Diversidad temática: {a3_4d_test['diversity']:.4f}")
print(f"  Usuarios evaluados:  {a3_4d_test['n']}")
print()
print(f"  (Referencia inner-val del ganador: NDCG={best['ndcg']:.4f}, R={best['recall']:.4f})")


## **7. Resumen comparativo final**

Esta tabla recoge los resultados de los tres algoritmos de este notebook,
junto con el modelo híbrido en sus dos calibraciones (3D y 4D). El
Algoritmo 4 (red neuronal) se evalúa en su propio notebook. Las cifras se
corresponden con las reportadas en la Tabla 3.5 de la memoria.

In [ ]:
resumen_final = pd.DataFrame([
    {'Algoritmo': 'Alg 1 — PPR',
     'Recall@10': np.mean(resultados['Algoritmo 1 (Topológico)']['recall']),
     'NDCG@10':   np.mean(resultados['Algoritmo 1 (Topológico)']['ndcg']),
     'Diversidad': '—'},
    {'Algoritmo': 'Alg 2 — Semántico',
     'Recall@10': np.mean(resultados['Algoritmo 2 (Semántico)']['recall']),
     'NDCG@10':   np.mean(resultados['Algoritmo 2 (Semántico)']['ndcg']),
     'Diversidad': f"{np.mean(resultados['Algoritmo 2 (Semántico)']['diversity']):.4f}"},
    {'Algoritmo': f"Alg 3 — 3D (grid α={a3_3d_test['alpha']}, β={a3_3d_test['beta']}, γ={a3_3d_test['gamma']})",
     'Recall@10': a3_3d_test['recall'],
     'NDCG@10':   a3_3d_test['ndcg'],
     'Diversidad': f"{a3_3d_test['diversity']:.4f}"},
    {'Algoritmo': f"Alg 3 — 4D (grid α={a3_4d_test['alpha']}, β={a3_4d_test['beta']}, γ={a3_4d_test['gamma']}, δ={a3_4d_test['delta']})",
     'Recall@10': a3_4d_test['recall'],
     'NDCG@10':   a3_4d_test['ndcg'],
     'Diversidad': f"{a3_4d_test['diversity']:.4f}"},
])

def fmt(x):
    return f"{x:.4f}" if isinstance(x, (int, float)) else x

print(resumen_final.to_string(index=False, formatters={'Recall@10': fmt, 'NDCG@10': fmt}))
print()
print('Nota: si el grid 4D converge a δ=0, las filas 3D y 4D coinciden.')

## **Ejemplo concreto por algoritmo**

Eligimos un usuario con histórico diverso (mensajes de 3-4 temáticas distintas) para que los ejemplos sean ilustrativos. Por ejemplo `william_fe3e1e_746`, que también tiene más de 6 interacciones en df_train y autores variados.

In [ ]:
ejemplo_user = 'william_fe3e1e_746'  # ≥6 interacciones y temáticas diversas

# Algoritmo 1
print(f"\n--- Algoritmo 1, PPR para {ejemplo_user} ---")
for rank, (u, s) in enumerate(get_ppr_recommendations(G_train, ejemplo_user, top_k=5,
                                                      ppr_cache=ppr_per_user), 1):
    info = df_users[df_users['username'] == u].iloc[0]
    print(f"  {rank}. {u}  ({info['name']}, {info['occupation']})  →  PPR={s:.5f}")

# Algoritmo 2
print(f"\n--- Algoritmo 2, Top-5 para {ejemplo_user} ---")
for rank, r in enumerate(get_semantic_recommendations(
        ejemplo_user, df_train, df_messages, embedding_map, model, top_k=5), 1):
    print(f"  {rank}. [{r['score']:.3f}] {r['author']}: {r['content'][:80]}...")

# Algoritmo 3 (3D), con la tripleta (α=0.4, β=0.3, γ=0.3)
print(f"\n--- Algoritmo 3 (3D, α=0.4, β=0.3, γ=0.3), Top-5 para {ejemplo_user} ---")
for rank, r in enumerate(get_hybrid_recommendations(
        ejemplo_user, df_train, df_messages, G_train,
        embedding_map, model, personality_dict,
        interests_dict=None, ppr_cache=ppr_per_user,
        alpha=0.4, beta=0.3, gamma=0.3, delta=0.0, top_k=5), 1):
    g, s, p, _ = r['scores']
    print(f"  {rank}. final={r['final_score']:.3f} (grafo={g:.2f}, sem={s:.2f}, pers={p:.2f}) "
          f"| {r['author']}: {r['content'][:60]}...")

# Algoritmo 3 (4D), con los pesos ganadores del grid 4D
print(f"\n--- Algoritmo 3 (4D, pesos del grid), Top-5 para {ejemplo_user} ---")
for rank, r in enumerate(get_hybrid_recommendations(
        ejemplo_user, df_train, df_messages, G_train,
        embedding_map, model, personality_dict,
        interests_dict=interests_dict, ppr_cache=ppr_per_user,
        alpha=a3_4d_test['alpha'], beta=a3_4d_test['beta'],
        gamma=a3_4d_test['gamma'], delta=a3_4d_test['delta'], top_k=5), 1):
    g, s, p, i = r['scores']
    print(f"  {rank}. final={r['final_score']:.3f} "
          f"(grafo={g:.2f}, sem={s:.2f}, pers={p:.2f}, int={i:.2f}) "
          f"| {r['author']}: {r['content'][:60]}...")

In [ ]:
# Baseline de contraste: A3 4D con pesos UNIFORMES (0.25 cada uno)
# No se eligen mirando df_test, por lo que no hay leakage, es solo una referencia
recall_def, ndcg_def, div_def = [], [], []
for user in usuarios_validos:
    true_texts = df_test[df_test['username'] == user]['message_text'].dropna().drop_duplicates().tolist()
    recs = get_hybrid_recommendations(
        user, df_train, df_messages, G_train,
        embedding_map, model, personality_dict,
        interests_dict=interests_dict,
        ppr_cache=ppr_per_user,
        alpha=0.25, beta=0.25, gamma=0.25, delta=0.25, top_k=10
    )
    if recs:
        rec_texts = [r['content'] for r in recs]
        ndcg_def.append(  calculate_ndcg(rec_texts,   true_texts))
        recall_def.append(calculate_recall(rec_texts, true_texts))
        div_def.append(   calculate_diversity(rec_texts, embedding_map, model))

print(f"A3 4D defecto (uniforme):  R={np.mean(recall_def):.4f}, "
      f"NDCG={np.mean(ndcg_def):.4f}, Div={np.mean(div_def):.4f}")